<a href="https://colab.research.google.com/github/kumarsirish/FDP-AGENENTIC-AI-RAG/blob/main/rag-adk-06/professor_assistant_adk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 Professor's AI Assistant — Groq SDK (No ADK Tool-Calling)

| Agent | Job |
|---|---|
| 🗂️ **Syllabus** | Answers questions about your syllabus |
| 📅 **Timetable** | Reads your timetable, finds free slots |
| 📝 **Quiz** | Generates quiz questions and MCQs |

> **Why this approach?** `llama-3.3-70b-versatile` generates malformed XML tool calls when
> routed through Google ADK's LiteLLM bridge. This version uses the **Groq SDK directly**
> with plain Python routing — no ADK tool-calling machinery, no XML bugs, same results.
>
> **Setup:** Add `GROQ_API_KEY` to Colab Secrets (🔑 sidebar). Run Cell 1 → restart → run all.

## Cell 1 — Install
> ⚠️ Restart runtime after this cell.

In [ ]:
!pip install -q groq PyPDF2 gdown
print('✅ Done — restart runtime now.')

## Cell 2 — Imports & API Key

In [ ]:
import os
from google.colab import userdata
from groq import Groq

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
client = Groq()
MODEL  = 'llama-3.3-70b-versatile'
print('✅ Ready.')

## Cell 3 — Load PDFs from Google Drive

In [ ]:
import PyPDF2, io, re, gdown

syllabus_drive_url  = 'https://drive.google.com/file/d/1K26k6ejc1MKzw7HegTUw6kxatj9BQmBv/view?usp=drive_link'
timetable_drive_url = 'https://drive.google.com/file/d/1v136u307B-zTp-S_jRg0kK8x7S3SprGJ/view?usp=drive_link'

SYLLABUS_TEXT  = ''
TIMETABLE_TEXT = ''

def extract_pdf(content):
    return '\n'.join(p.extract_text() or '' for p in PyPDF2.PdfReader(io.BytesIO(content)).pages)

def get_drive_file_id(url):
    m = re.search(r'/d/([a-zA-Z0-9_-]+)', url)
    return m.group(1) if m else (_ for _ in ()).throw(ValueError(f'Bad URL: {url}'))

print('📄 Downloading SYLLABUS...')
try:
    gdown.download(id=get_drive_file_id(syllabus_drive_url), output='syllabus.pdf', quiet=True)
    SYLLABUS_TEXT = extract_pdf(open('syllabus.pdf','rb').read())
    print(f'✅ Syllabus loaded ({len(SYLLABUS_TEXT)} chars)')
except Exception as e:
    print(f'⚠️  Skipped: {e}')

print('\n🗓️  Downloading TIMETABLE...')
try:
    gdown.download(id=get_drive_file_id(timetable_drive_url), output='timetable.pdf', quiet=True)
    TIMETABLE_TEXT = extract_pdf(open('timetable.pdf','rb').read())
    print(f'✅ Timetable loaded ({len(TIMETABLE_TEXT)} chars)')
except Exception as e:
    print(f'⚠️  Skipped: {e}')

## Cell 4 — Pure-Python Tool Logic
> Same logic as before — just plain functions, no ADK `ToolContext`.

In [ ]:
# ── session state dict (replaces ADK session) ─────────────
STATE = {'quizzes': []}
if SYLLABUS_TEXT:  STATE['syllabus']  = SYLLABUS_TEXT
if TIMETABLE_TEXT: STATE['timetable'] = TIMETABLE_TEXT

def get_syllabus() -> str:
    text = STATE.get('syllabus', '')
    return text if text.strip() else 'No syllabus uploaded.'

def search_syllabus(topic: str) -> str:
    text = STATE.get('syllabus', '')
    if not text.strip(): return 'No syllabus uploaded.'
    matches = [l for l in text.splitlines() if topic.lower() in l.lower()]
    return ('\n'.join(matches[:20])) if matches else f"'{topic}' not found."

def get_timetable() -> str:
    text = STATE.get('timetable', '')
    if not text.strip(): return 'No timetable uploaded.'
    lines = [l for l in text.splitlines() if ' | ' in l]
    return '\n'.join(lines) if lines else text

def search_timetable(query: str) -> str:
    text = STATE.get('timetable', '')
    if not text.strip(): return 'No timetable uploaded.'
    lines = [l for l in text.splitlines() if ' | ' in l]
    matches = [l for l in lines if query.lower() in l.lower()]
    return '\n'.join(matches) if matches else f"No entries for '{query}'."

def get_free_slots(professor_name: str) -> str:
    text = STATE.get('timetable', '')
    if not text.strip(): return 'No timetable uploaded.'
    all_slots = ['9:00-10:00','10:00-11:00','11:00-12:00','2:00-3:00','3:00-4:00','4:00-5:00']
    days = ['Monday','Tuesday','Wednesday','Thursday','Friday']
    lines = [l for l in text.splitlines() if ' | ' in l]
    busy = {}
    for line in lines:
        parts = [p.strip() for p in line.split('|')]
        if len(parts) == 4:
            day, slot, subject, prof = parts
            if professor_name.lower() in prof.lower() and 'Break' not in subject:
                busy.setdefault(day, []).append(slot)
    if not any(busy.values()):
        return f"No professor matching '{professor_name}' found."
    result = [f"Schedule for '{professor_name}':"]
    for day in days:
        bs = busy.get(day, [])
        fs = [s for s in all_slots if s not in bs]
        result += [f"  {day}:", f"    Busy: {', '.join(bs) or 'None'}", f"    Free: {', '.join(fs) or 'None'}"]
    return '\n'.join(result)

def save_quiz(title: str, content: str) -> str:
    STATE['quizzes'].append({'title': title, 'content': content})
    return f"Saved '{title}'. Total: {len(STATE['quizzes'])}."

def list_quizzes() -> str:
    q = STATE['quizzes']
    return '\n'.join(f"{i+1}. {x['title']}" for i,x in enumerate(q)) if q else 'No quizzes saved.'

print('✅ Tools defined.')

## Cell 5 — Router & `ask()`
> Keyword-based router picks the right system prompt + context. One Groq call per question — no tool-calling API needed.

In [ ]:
def _route(question: str) -> tuple[str, str]:
    """Return (agent_name, system_prompt_with_context) for the question."""
    q = question.lower()

    # ── Timetable agent ──────────────────────────────────────
    if any(w in q for w in ['schedule','timetable','free','office hour','available','slot','monday',
                             'tuesday','wednesday','thursday','friday','class time','when do i']):
        tt = get_timetable()
        return '📅 timetable_agent', (
            'You are a timetable assistant. Answer using ONLY the timetable data below.\n'
            'For free-slot questions, list busy and free slots per day.\n'
            'For schedule questions, list the matching rows.\n\n'
            f'TIMETABLE:\n{tt}'
        )

    # ── Quiz agent ───────────────────────────────────────────
    if any(w in q for w in ['quiz','mcq','question','test','exam','generate']):
        syl = get_syllabus()
        return '📝 quiz_agent', (
            'You are a quiz generator. Create quiz questions with answer keys.\n'
            'After generating, output: SAVE_QUIZ: <title> | <full quiz text>\n\n'
            f'SYLLABUS:\n{syl}'
        )

    # ── Syllabus agent (default) ─────────────────────────────
    syl = get_syllabus()
    return '🗂️ syllabus_agent', (
        'You are a syllabus assistant. Answer using ONLY the syllabus data below.\n\n'
        f'SYLLABUS:\n{syl}'
    )


def ask(question: str):
    print(f'\n\U0001f9d1\u200d\U0001f3eb {question}\n' + '─'*55)
    agent_name, system_prompt = _route(question)
    print(f'  → [{agent_name}]')

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system',  'content': system_prompt},
            {'role': 'user',    'content': question},
        ],
        temperature=0.2,
        max_tokens=1024,
    )
    answer = resp.choices[0].message.content.strip()

    # ── auto-save quizzes if agent used SAVE_QUIZ protocol ───
    if 'SAVE_QUIZ:' in answer:
        for line in answer.splitlines():
            if line.startswith('SAVE_QUIZ:'):
                parts = line[len('SAVE_QUIZ:'):].split('|', 1)
                if len(parts) == 2:
                    result = save_quiz(parts[0].strip(), parts[1].strip())
                    print(f'  💾 {result}')
        answer = answer.split('SAVE_QUIZ:')[0].strip()

    print(answer)

print('✅ Ready.')

---
## Demo — Syllabus

In [ ]:
ask('Summarise the course syllabus.')

In [ ]:
ask('Is github covered? Which section?')

## Demo — Timetable

In [ ]:
ask('I am Dr. Rajesh Kumar. What is my schedule on Monday?')

In [ ]:
ask('When do I have free time for office hours?')

## Demo — Quiz

In [ ]:
ask('Create a 2-question MCQ quiz based on the devops syllabus.')

## Demo — Multi-topic

In [ ]:
ask('Read my syllabus and create a quiz on the first topic. List the Quiz')

## Saved Quizzes

In [ ]:
print(list_quizzes())